# Notebook 15b — Out-of-Sample Random-Forest-Predictions

**CRISP-DM Phase:** Modeling (Korrekturschritt)  
**Skript-Bezug:** Kapitel 6.2 (Regression), Kapitel 6.1.3 (Modell-Evaluierung)

**Motivation:** Die in Notebook 15 generierten Predictions wurden auf dem gesamten Datensatz (2020–2025) trainiert. Bei Verwendung im Backtest (Notebook 16) führt dies zu Look-Ahead-Bias: Das Modell "kennt" die zukünftigen Marktentwicklungen, was zu unrealistisch hohen Backtest-Renditen führte (+485 % Total Return).

**Korrekturansatz:** Strikte Trennung von Trainings- und Backtest-Periode:
- **Trainings-Periode:** 01.01.2020 – 31.12.2023 (~4 Jahre)
- **Backtest-Periode:** 01.01.2024 – 31.12.2025 (~22 Monate)

Das Modell wird ausschließlich auf Trainingsdaten gefittet und liefert anschließend Predictions für die Backtest-Periode, ohne dass es die dortigen Marktbewegungen je gesehen hat.

**Hyperparameter:** Aus Notebook 15 übernommen, da diese auf Walk-Forward-Splits ermittelt wurden und kein Look-Ahead-Bias enthalten.

## 1. Setup & Datenimport

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

BASE = Path("..")
PROC = BASE / "data" / "processed"

RANDOM_STATE = 42

# Daten laden
basket = pd.read_parquet(PROC / "basket_features.parquet")
basket = basket.sort_values(["Ticker", "Date"]).copy()

print(f"Datensatz: {basket.shape}")
print(f"Zeitraum:  {basket['Date'].min().date()} bis {basket['Date'].max().date()}")
print(f"Spalten:   {list(basket.columns)}")

Datensatz: (120473, 25)
Zeitraum:  2020-01-03 bis 2025-11-26
Spalten:   ['Date', 'Ticker', 'Adj_Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Index_Name', 'Daily_Return', 'Log_Return', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'Volatility_30d', 'MA_50', 'MA_200', 'RSI_14', 'Forward_Return_1M', 'Sector', 'Industry', 'Market_Cap', 'P_E_Ratio', 'Dividend_Yield', 'Price_To_Book']


## 2. Feature- und Zielvariablen-Setup

Konsistent zu Notebook 15: gleiche Features, `Forward_Return_1W` als Zielvariable. Falls die Spalte noch nicht im Parquet enthalten ist, wird sie hier berechnet.

In [2]:
FEATURE_COLS = [
    "Daily_Return", "Momentum_3M", "Momentum_6M", "Momentum_12M",
    "Volatility_30d", "MA_50", "MA_200", "RSI_14",
]
TARGET_COL = "Forward_Return_1W"

# Forward_Return_1W berechnen, falls noch nicht vorhanden
if TARGET_COL not in basket.columns:
    basket[TARGET_COL] = (
        basket.groupby("Ticker")["Close"]
        .transform(lambda s: s.shift(-5) / s - 1)
    )
    print(f"'{TARGET_COL}' neu berechnet (5-Tage-Forward-Return).")
else:
    print(f"'{TARGET_COL}' bereits im Datensatz vorhanden.")

modeling_cols = FEATURE_COLS + [TARGET_COL, "Date", "Ticker"]
df_full = basket[modeling_cols].dropna().sort_values("Date").reset_index(drop=True)

print(f"\nVollständiger Modeling-Datensatz: {len(df_full):,} Zeilen")
print(f"Zeitraum: {df_full['Date'].min().date()} bis {df_full['Date'].max().date()}")
print(f"Features: {FEATURE_COLS}")
print(f"Target:   {TARGET_COL}")

'Forward_Return_1W' neu berechnet (5-Tage-Forward-Return).

Vollständiger Modeling-Datensatz: 120,073 Zeilen
Zeitraum: 2020-01-03 bis 2025-11-19
Features: ['Daily_Return', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'Volatility_30d', 'MA_50', 'MA_200', 'RSI_14']
Target:   Forward_Return_1W


## 3. Zeitliche Trennung in Training und Test

**Strikte Trennung am 01.01.2024.** Alle Daten vor diesem Datum sind Training, alle ab diesem Datum sind Test/Backtest. Diese Trennung ist die zentrale methodische Korrektur gegenüber Notebook 15.

In [3]:
SPLIT_DATE = pd.Timestamp("2024-01-01")

df_train = df_full[df_full["Date"] < SPLIT_DATE].copy()
df_test = df_full[df_full["Date"] >= SPLIT_DATE].copy()

print(f"Training-Periode: {df_train['Date'].min().date()} bis {df_train['Date'].max().date()}")
print(f"Test-Periode:     {df_test['Date'].min().date()} bis {df_test['Date'].max().date()}")
print()
print(f"Training:  {len(df_train):,} Zeilen ({len(df_train) / len(df_full) * 100:.1f} %)")
print(f"Test:      {len(df_test):,} Zeilen ({len(df_test) / len(df_full) * 100:.1f} %)")
print()
print(f"Aktien im Training: {df_train['Ticker'].nunique()}")
print(f"Aktien im Test:     {df_test['Ticker'].nunique()}")

Training-Periode: 2020-01-03 bis 2023-12-29
Test-Periode:     2024-01-02 bis 2025-11-19

Training:  81,594 Zeilen (68.0 %)
Test:      38,479 Zeilen (32.0 %)

Aktien im Training: 80
Aktien im Test:     80


## 4. Random Forest auf Trainings-Periode fitten

Hyperparameter aus Notebook 15 (BayesSearchCV):
- `max_depth`: 3
- `max_features`: 0.32
- `min_samples_leaf`: 13
- `n_estimators`: 51

In [4]:
# Hyperparameter aus Notebook 15 (BayesSearchCV-Ergebnis)
best_params_from_notebook15 = {
    "max_depth": 3,
    "max_features": 0.31832990549524615,
    "min_samples_leaf": 13,
    "n_estimators": 51,
}

rf_oos = RandomForestRegressor(
    **best_params_from_notebook15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

X_train = df_train[FEATURE_COLS].values
y_train = df_train[TARGET_COL].values

print("Trainiere Random Forest auf Trainings-Periode (2020–2023)...")
rf_oos.fit(X_train, y_train)
print("Fertig.")

# In-Sample R² nur zur Diagnose (zeigt Overfitting-Grad)
y_train_pred = rf_oos.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
print(f"\nIn-Sample R² (Training):  {r2_train:.5f}")
print("(In-Sample R² ist immer optimistisch; Bewertung erfolgt auf Test-Daten)")

Trainiere Random Forest auf Trainings-Periode (2020–2023)...
Fertig.

In-Sample R² (Training):  0.02235
(In-Sample R² ist immer optimistisch; Bewertung erfolgt auf Test-Daten)


## 5. Predictions für Test-Periode generieren

In [5]:
X_test = df_test[FEATURE_COLS].values
y_test = df_test[TARGET_COL].values

# Out-of-Sample Predictions
y_test_pred = rf_oos.predict(X_test)

# Out-of-Sample Metriken
r2_test = r2_score(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
correlation = np.corrcoef(y_test, y_test_pred)[0, 1]

print("--- Out-of-Sample Performance (Test-Periode 2024–2025) ---")
print(f"R²:            {r2_test:.5f}")
print(f"MSE:           {mse_test:.5f}")
print(f"MAE:           {mae_test:.5f}")
print(f"Pearson r:     {correlation:.4f}")

--- Out-of-Sample Performance (Test-Periode 2024–2025) ---
R²:            -0.00760
MSE:           0.00233
MAE:           0.03401
Pearson r:     -0.0052


## 6. Predictions-DataFrame zusammensetzen und speichern

Die Predictions enthalten Ticker, Datum, vorhergesagte Rendite und tatsächliche Rendite — strukturiert für die Verwendung im Backtest (Notebook 16).

In [6]:
df_predictions_oos = df_test[["Date", "Ticker", TARGET_COL]].copy()
df_predictions_oos = df_predictions_oos.rename(columns={TARGET_COL: "Actual_Return"})
df_predictions_oos["Predicted_Return_Bayes"] = y_test_pred

print(f"Predictions-DataFrame: {df_predictions_oos.shape}")
print(f"Spalten: {list(df_predictions_oos.columns)}")
print(f"\nErste 5 Zeilen:")
print(df_predictions_oos.head().to_string(index=False))

oos_path = PROC / "predictions_bayes_oos.parquet"
df_predictions_oos.to_parquet(oos_path, index=False)
print(f"\nGespeichert: {oos_path}")

Predictions-DataFrame: (38479, 4)
Spalten: ['Date', 'Ticker', 'Actual_Return', 'Predicted_Return_Bayes']

Erste 5 Zeilen:
      Date  Ticker  Actual_Return  Predicted_Return_Bayes
2024-01-02  JEN.DE      -0.058148                0.001401
2024-01-02  MBG.DE       0.007421                0.001045
2024-01-02  ADS.DE      -0.022610                0.000791
2024-01-02 EOAN.DE       0.048811                0.001401
2024-01-02  MRK.DE       0.003505                0.000791

Gespeichert: ../data/processed/predictions_bayes_oos.parquet


## 7. Vergleich: In-Sample vs. Out-of-Sample

Quantifizierung des Look-Ahead-Effekts: Wie viel "schummelt" das ursprüngliche In-Sample-Setup?

In [7]:
try:
    df_pred_insample = pd.read_parquet(PROC / "predictions_bayes.parquet")

    df_pred_is_test = df_pred_insample[df_pred_insample["Date"] >= SPLIT_DATE].copy()

    comparison = df_pred_is_test[["Date", "Ticker", "Predicted_Return_Bayes"]].merge(
        df_predictions_oos[["Date", "Ticker", "Predicted_Return_Bayes", "Actual_Return"]],
        on=["Date", "Ticker"],
        how="inner",
        suffixes=("_IS", "_OOS"),
    )

    print(f"Vergleich auf Test-Periode (2024–2025):")
    print(f"Anzahl Beobachtungen: {len(comparison):,}")
    print()

    r2_is = r2_score(comparison["Actual_Return"], comparison["Predicted_Return_Bayes_IS"])
    r2_oos_cmp = r2_score(comparison["Actual_Return"], comparison["Predicted_Return_Bayes_OOS"])
    corr_is = np.corrcoef(comparison["Actual_Return"], comparison["Predicted_Return_Bayes_IS"])[0, 1]
    corr_oos = np.corrcoef(comparison["Actual_Return"], comparison["Predicted_Return_Bayes_OOS"])[0, 1]

    print(f"R² In-Sample-Predictions  auf Test-Daten: {r2_is:.5f}")
    print(f"R² Out-of-Sample-Predictions auf Test-Daten: {r2_oos_cmp:.5f}")
    print()
    print(f"Korrelation In-Sample-Pred  mit Actual: {corr_is:.4f}")
    print(f"Korrelation OOS-Pred        mit Actual: {corr_oos:.4f}")

except FileNotFoundError:
    print("predictions_bayes.parquet nicht gefunden — Vergleich übersprungen.")
    print("Nur Out-of-Sample-Metriken verfügbar:")
    print(f"  R²:        {r2_test:.5f}")
    print(f"  Pearson r: {correlation:.4f}")

Vergleich auf Test-Periode (2024–2025):
Anzahl Beobachtungen: 38,479

R² In-Sample-Predictions  auf Test-Daten: 0.00318
R² Out-of-Sample-Predictions auf Test-Daten: -0.00760

Korrelation In-Sample-Pred  mit Actual: 0.0579
Korrelation OOS-Pred        mit Actual: -0.0052


## 8. Methodische Erkenntnisse

### Bedeutung dieser Korrektur

Die Out-of-Sample-Variante ist methodisch korrekt: Das Modell trifft Vorhersagen, ohne die Zukunft zu kennen. Realistische R²-Werte für Out-of-Sample-Aktien-Renditeprognosen liegen typischerweise bei 0,01–0,05 oder leicht negativ — abhängig von Marktphase und Feature-Qualität.

### Konsequenz für Notebook 16

Der Backtest in Notebook 16 wird auf `predictions_bayes_oos.parquet` umgestellt. Das Feld `Predicted_Return_Bayes` ist identisch zu `Predicted_Return_OOS` — so ist Notebook 16 ohne Code-Änderungen kompatibel. Erwartet wird ein deutlich realistischeres Ergebnis: moderate Outperformance (positiv oder negativ), möglicherweise nicht statistisch signifikant.

### Dokumentation für die Abschlussarbeit

Der ursprüngliche In-Sample-Backtest (+485 % Outperformance) bleibt als Beispiel für **Look-Ahead-Bias** in der Limitations-Sektion dokumentiert. Dies ist ein lehrreiches Beispiel für methodische Sorgfalt: Quantitative Strategien können in In-Sample-Backtests beeindruckend aussehen und in Out-of-Sample-Tests komplett versagen.